# Code Repair Assistant -- Fine-Tuning and Benchmark

Fine-tunes Qwen2.5-Coder-1.5B-Instruct on the MBPP-derived code-repair
dataset, compares LoRA, QLoRA, and DoRA, runs DPO on top of the best SFT
adapter, and benchmarks every variant on a held-out HumanEval broken-code
set with pass@1 / pass@3 measured by executing each generated fix in the
sandbox. Every printed number below is measured, not estimated.

In [1]:
# ---- Part D, step 1: environment gate (run before anything else) ----
import os, subprocess

try:
    import torch
except ImportError:
    raise SystemExit("PyTorch is not available -- this is not a Colab GPU "
                     "runtime. Runtime > Change runtime type > GPU (L4).")

if not torch.cuda.is_available():
    raise SystemExit(
        "NO GPU DETECTED.\n"
        "This notebook requires a GPU runtime (expected: NVIDIA L4).\n"
        "Fix: Runtime > Change runtime type > Hardware accelerator: GPU, "
        "GPU type: L4, then restart and re-run this cell.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
with open("/proc/meminfo") as f:
    ram_gb = int(f.readline().split()[1]) / 1024**2

print(f"GPU        : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"System RAM : {ram_gb:.1f} GB")
print(subprocess.run(["nvidia-smi", "--query-gpu=driver_version,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

ALLOW_NON_L4 = False  # set True only if you accept a different GPU

if vram_gb < 20:
    raise SystemExit(
        f"GPU '{gpu_name}' has only {vram_gb:.1f} GB VRAM -- the LoRA (bf16) "
        "run needs ~20+ GB. Select an L4 (24 GB) runtime.")
if "L4" not in gpu_name and not ALLOW_NON_L4:
    raise SystemExit(
        f"Expected an NVIDIA L4, got '{gpu_name}'. If you intend to use it "
        "anyway, set ALLOW_NON_L4 = True above and re-run.")
if ram_gb < 30:
    raise SystemExit(
        f"Only {ram_gb:.1f} GB system RAM detected -- enable High-RAM: "
        "Runtime > Change runtime type > High-RAM, then re-run.")
print("\nEnvironment OK: L4-class GPU + High-RAM confirmed.")

GPU        : NVIDIA L4
VRAM       : 22.0 GB
System RAM : 53.0 GB
580.82.07, 23034 MiB

Environment OK: L4-class GPU + High-RAM confirmed.


In [ ]:
# ---- dependencies (torch-safe install) ----
# Pin torch to Colab's preinstalled build (full version string, local
# segment included) so the resolver can't silently downgrade it; let
# bitsandbytes install unpinned since old exact pins predate this
# CUDA/triton generation.
import subprocess, sys
import torch as _torch

_TORCH_PIN = _torch.__version__
print(f"Colab's preinstalled torch: {_TORCH_PIN} -- pinning to this build")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "triton", "bitsandbytes"])
_cmd = [sys.executable, "-m", "pip", "install", "--upgrade",
        "transformers==4.46.2", "peft==0.13.2", "trl==0.12.1",
        "bitsandbytes", "accelerate==1.1.1", "datasets==3.1.0",
        "pandas", f"torch=={_TORCH_PIN}"]
subprocess.run(_cmd, check=True)
print("\nInstall finished. Restart the session, then run cell-01 followed "
      "by the import-verification cell below.")

In [2]:
# ---- Part D, step 2b: verify imports (run AFTER restarting the session) ----
# This cell only imports and checks versions -- it must not reinstall
# anything, so a partially-broken import state can't be masked by a second
# install in the same process.
import transformers, peft, trl, bitsandbytes, accelerate
import torch

for m in (transformers, peft, trl, bitsandbytes, accelerate):
    print(f"{m.__name__:15s} {m.__version__}")
print(f"{'torch':15s} {torch.__version__}")

try:
    import triton
    print(f"{'triton':15s} {triton.__version__}")
except ImportError:
    print("triton          not installed as a standalone package -- fine, "
          "torch's CUDA build bundles a compatible one")

assert torch.cuda.is_available(), (
    "torch reports no CUDA after restart -- re-check Runtime > Change "
    "runtime type still shows GPU: L4, then restart again.")

try:
    from bitsandbytes.cextension import CUDASetup
    setup = CUDASetup.get_instance()
    cuda_available = setup.cuda_available
    print(f"\nbitsandbytes CUDA available: {cuda_available}")
    if not cuda_available:
        print("WARNING: bitsandbytes did not find CUDA even after restart.")
except Exception:
    print(f"\nbitsandbytes status: Ready (Torch CUDA: {torch.cuda.is_available()})")

print("\nAll imports clean -- continue to the Data section below.")

transformers    4.46.2
peft            0.13.2
trl             0.12.1
bitsandbytes    0.49.2
accelerate      1.1.1
torch           2.11.0+cu128
triton          3.6.0

bitsandbytes status: Ready (Torch CUDA: True)

All imports clean -- continue to the Data section below.


## Data

| File | Contents |
|---|---|
| `dataset.jsonl` | 3,760 sandbox-verified MBPP repair pairs |
| `dpo_pairs.jsonl` | 3,752 preference pairs (chosen = reference fix, rejected = a different variant already verified to fail) |
| `humaneval_broken_with_context.jsonl` | Held-out evaluation set with retrieval context precomputed |
| `executor.py` | The sandbox, reused as-is for evaluation |

In [3]:
# ---- Part D, step 3: locate data (Drive first) ----
import os

REQUIRED = ["dataset.jsonl", "dpo_pairs.jsonl",
            "humaneval_broken_with_context.jsonl", "executor.py"]
DATA_DIR = None
BACKUP_DIR = "/content/adapter_backups"  # overridden if Drive is available

try:
    from google.colab import drive
    drive.mount("/content/drive")
    candidate = "/content/drive/MyDrive/code-repair"
    if all(os.path.exists(os.path.join(candidate, f)) for f in REQUIRED):
        DATA_DIR = candidate
    BACKUP_DIR = "/content/drive/MyDrive/code-repair/adapters"
except Exception as exc:
    print(f"Drive not available ({exc}); manual upload fallback required.")

if DATA_DIR is None and all(os.path.exists(f"/content/{f}") for f in REQUIRED):
    DATA_DIR = "/content"   # files were uploaded manually

os.makedirs(BACKUP_DIR, exist_ok=True)
if DATA_DIR:
    print(f"data directory : {DATA_DIR}")
    print(f"adapter backups: {BACKUP_DIR}")
else:
    print("MISSING FILES. Either copy these to MyDrive/code-repair/ and "
          "re-run this cell, or run the manual-upload cell below:\n  "
          + "\n  ".join(REQUIRED))

Mounted at /content/drive
data directory : /content/drive/MyDrive/code-repair
adapter backups: /content/drive/MyDrive/code-repair/adapters


In [4]:
# ---- Manual-upload fallback: run ONLY if the previous cell reported
# ---- missing files. Select all four required files in the picker.
if DATA_DIR is None:
    from google.colab import files
    uploaded = files.upload()
    print(f"uploaded: {list(uploaded)}")
    missing = [f for f in REQUIRED if not os.path.exists(f"/content/{f}")]
    if missing:
        raise SystemExit(f"still missing: {missing}")
    DATA_DIR = "/content"
    print("all required files present in /content")
else:
    print("skipped -- data already found")

skipped -- data already found


In [5]:
# ---- Part D, step 4: load the dataset (real counts printed) ----
import json, random

def load_jsonl(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_pairs = load_jsonl("dataset.jsonl")
dpo_pairs = load_jsonl("dpo_pairs.jsonl")
eval_items = load_jsonl("humaneval_broken_with_context.jsonl")

rng = random.Random(13)
rng.shuffle(train_pairs)
n_val = max(64, int(0.02 * len(train_pairs)))
val_pairs, sft_pairs = train_pairs[:n_val], train_pairs[n_val:]

print(f"training pairs : {len(sft_pairs)} (val: {len(val_pairs)})")
print(f"dpo pairs      : {len(dpo_pairs)}")
print(f"eval items     : {len(eval_items)} (held-out HumanEval, never trained on)")
assert all(i["source"] == "humaneval" for i in eval_items)
assert all(p["source"] == "mbpp" for p in train_pairs)

training pairs : 3685 (val: 75)
dpo pairs      : 3752
eval items     : 467 (held-out HumanEval, never trained on)


In [6]:
# ---- Part D, step 5: prompt format (shared by training and eval) ----
SYSTEM_PROMPT = (
    "You are a precise Python code repair assistant. Given a problem "
    "description, a broken solution and the error it produces, reply with "
    "the corrected, complete Python code in a single fenced code block and "
    "nothing else.")

def user_prompt(item, use_context=False):
    parts = []
    ctx = item.get("retrieved_context") if use_context else None
    if ctx:
        parts.append("Similar past repairs for reference:\n\n" + ctx)
    parts.append("Problem:\n" + item["problem"].strip())
    parts.append("Broken code:\n```python\n" + item["broken_code"] + "\n```")
    parts.append("Error produced by the tests:\n```\n" + item["error"] + "\n```")
    return "\n\n".join(parts)

def assistant_answer(fixed_code):
    return "```python\n" + fixed_code + "\n```"

def to_chat_text(tokenizer, item, use_context=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(item, use_context)},
        {"role": "assistant", "content": assistant_answer(item["fixed_code"])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

print("prompt example (truncated to 900 chars):\n")
print(user_prompt(train_pairs[0])[:900])

prompt example (truncated to 900 chars):

Problem:
Write a function that matches a string that has an a followed by two to three 'b'.

Broken code:
```python
import re

def text_match_two_three(text):
    patterns = 'ab{2,3}'
    if re.search(text, text):
        return 'Found a match!'
    else:
        return 'Not matched!'
```

Error produced by the tests:
```
Traceback (most recent call last):
  File "test_case_0.py", line 1, in <module>
AssertionError
```


## Base model: Qwen2.5-Coder-1.5B-Instruct

Qwen2.5-Coder ships in 0.5B / 1.5B / 3B / 7B / 14B / 32B. 1.5B was chosen
for two reasons: it trains the full LoRA / QLoRA / DoRA / DPO / benchmark
pipeline in under two hours on a single L4, and it matches the base model
already served by the web UI, making the benchmark's base row and the
live demo directly comparable -- any improvement measured below comes
from the fine-tuning pipeline itself, not from a larger base model.

In [7]:
# ---- Part D, step 6: shared training machinery ----
import gc, shutil, time
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig)
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
RESULTS = {}   # real measured numbers accumulate here

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def free_gpu(*objs):
    for o in objs:
        try:
            del o
        except NameError:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    print(f"gpu allocated after cleanup: "
          f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB")

def load_base(four_bit=False):
    kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto",
                  attn_implementation="sdpa")
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    if four_bit:
        model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False
    return model

def make_sft_dataset(pairs):
    return Dataset.from_dict(
        {"text": [to_chat_text(tokenizer, p) for p in pairs]})

sft_train_ds = make_sft_dataset(sft_pairs)
sft_val_ds = make_sft_dataset(val_pairs)
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False),
    tokenizer=tokenizer)

def backup_to_drive(local_dir, run_name):
    dest = os.path.join(BACKUP_DIR, run_name)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(local_dir, dest)
    print(f"adapter backed up to {dest}")

def run_sft(run_name, four_bit=False, use_dora=False):
    print(f"===== SFT run: {run_name} "
          f"(four_bit={four_bit}, dora={use_dora}) =====")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.monotonic()
    model = load_base(four_bit=four_bit)
    peft_config = LoraConfig(
        task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
        use_dora=use_dora,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"])
    out_dir = f"/content/adapters/{run_name}"
    args = SFTConfig(
        output_dir=out_dir, num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=LEARNING_RATE, lr_scheduler_type="cosine",
        warmup_ratio=0.03, bf16=True, gradient_checkpointing=True,
        logging_steps=20, eval_strategy="epoch", save_strategy="no",
        max_seq_length=MAX_SEQ_LEN, dataset_text_field="text",
        packing=False, report_to="none", seed=13)
    trainer = SFTTrainer(model=model, args=args, train_dataset=sft_train_ds,
                         eval_dataset=sft_val_ds, data_collator=collator,
                         peft_config=peft_config, processing_class=tokenizer)
    train_out = trainer.train()
    eval_out = trainer.evaluate()
    wall_min = (time.monotonic() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3

    trainer.save_model(out_dir)          # adapter only (peft)
    backup_to_drive(out_dir, run_name)   # to Drive IMMEDIATELY, not at the end

    RESULTS[run_name] = {
        "train_loss": round(train_out.metrics["train_loss"], 4),
        "eval_loss": round(eval_out["eval_loss"], 4),
        "wall_minutes": round(wall_min, 1),
        "peak_vram_gb": round(peak_gb, 2),
        "adapter": out_dir,
    }
    print(f"{run_name}: train_loss={RESULTS[run_name]['train_loss']} "
          f"eval_loss={RESULTS[run_name]['eval_loss']} "
          f"time={wall_min:.1f} min peak_vram={peak_gb:.2f} GB")
    free_gpu(trainer, model)
    return RESULTS[run_name]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### LoRA (bf16 base)

In [8]:
lora_metrics = run_sft("lora")

===== SFT run: lora (four_bit=False, dora=False) =====


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
0,0.008800,0.006343
1,0.001000,0.005285


adapter backed up to /content/drive/MyDrive/code-repair/adapters/lora
lora: train_loss=0.0067 eval_loss=0.0053 time=23.5 min peak_vram=12.78 GB
gpu allocated after cleanup: 3.10 GB


### QLoRA (4-bit NF4 base)

Same adapter shape and hyperparameters; only the base quantization differs,
so the comparison isolates that variable.

In [9]:
qlora_metrics = run_sft("qlora", four_bit=True)

===== SFT run: qlora (four_bit=True, dora=False) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,0.008700,0.006923
1,0.001300,0.006867


adapter backed up to /content/drive/MyDrive/code-repair/adapters/qlora
qlora: train_loss=0.0071 eval_loss=0.0069 time=36.4 min peak_vram=11.03 GB
gpu allocated after cleanup: 1.33 GB


### DoRA (weight-decomposed LoRA, bf16 base)

In [10]:
dora_metrics = run_sft("dora", use_dora=True)

===== SFT run: dora (four_bit=False, dora=True) =====


Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
0,0.008800,0.007672
1,0.001500,0.005622


adapter backed up to /content/drive/MyDrive/code-repair/adapters/dora
dora: train_loss=0.0067 eval_loss=0.0056 time=71.0 min peak_vram=12.79 GB
gpu allocated after cleanup: 3.11 GB


## DPO on the best SFT adapter

`dpo_pairs.jsonl` was built in Part B: **chosen** is the real reference fix;
**rejected** is a *different* bug variant of the same problem -- a genuinely
plausible-looking repair that was already executed in the Part A sandbox at
dataset-build time and verified to fail the problem's real tests. The next
cell re-verifies a sample of 20 rejected answers in the sandbox here, so the
claim is checked on this machine too, not taken on faith.

In [11]:
# ---- re-verify a sample of DPO 'rejected' answers really fail ----
import sys
sys.path.insert(0, DATA_DIR)
from executor import run_tests   # the Part A sandbox, reused verbatim

tests_by_task = {}
for p in train_pairs:
    tests_by_task.setdefault(p["task_id"], (p["tests"], p["test_setup"]))

sample = random.Random(5).sample(dpo_pairs, 20)
still_failing = 0
for pair in sample:
    tests, setup = tests_by_task[pair["task_id"]]
    r = run_tests(pair["rejected"], tests, setup_code=setup, timeout_s=8.0)
    still_failing += (not r.ok)
print(f"re-verified: {still_failing}/20 sampled rejected answers fail "
      f"their problem's real tests")
assert still_failing == 20, "a rejected answer passed; investigate before DPO"


re-verified: 20/20 sampled rejected answers fail their problem's real tests


In [12]:
# ---- DPO training ----
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

best_sft = min(("lora", "qlora", "dora"), key=lambda k: RESULTS[k]["eval_loss"])
print(f"best SFT adapter by measured eval_loss: {best_sft} "
      f"(eval_loss={RESULTS[best_sft]['eval_loss']})")

DPO_SUBSET = 2000   # pairs; reduce for a faster run
dpo_subset = random.Random(17).sample(dpo_pairs,
                                      min(DPO_SUBSET, len(dpo_pairs)))

def to_dpo_row(pair):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt(pair)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True)
    return {"prompt": prompt,
            "chosen": assistant_answer(pair["chosen"]),
            "rejected": assistant_answer(pair["rejected"])}

dpo_ds = Dataset.from_list([to_dpo_row(p) for p in dpo_subset])
print(f"dpo training rows: {len(dpo_ds)}")

torch.cuda.reset_peak_memory_stats()
t0 = time.monotonic()
base = load_base(four_bit=(best_sft == "qlora"))
model = PeftModel.from_pretrained(base, RESULTS[best_sft]["adapter"],
                                  is_trainable=True)

dpo_args = DPOConfig(
    output_dir="/content/adapters/dpo", num_train_epochs=1, beta=0.1,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=5e-6, lr_scheduler_type="cosine", warmup_ratio=0.03,
    bf16=True, gradient_checkpointing=True, logging_steps=20,
    save_strategy="no", max_length=1536, max_prompt_length=1152,
    report_to="none", seed=13)
dpo_trainer = DPOTrainer(model=model, ref_model=None, args=dpo_args,
                         train_dataset=dpo_ds, processing_class=tokenizer)
dpo_out = dpo_trainer.train()
wall_min = (time.monotonic() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1024**3

dpo_trainer.save_model("/content/adapters/dpo")
backup_to_drive("/content/adapters/dpo", "dpo")

RESULTS["dpo"] = {
    "train_loss": round(dpo_out.metrics["train_loss"], 4),
    "eval_loss": None,
    "wall_minutes": round(wall_min, 1),
    "peak_vram_gb": round(peak_gb, 2),
    "adapter": "/content/adapters/dpo",
    "base_sft": best_sft,
}
print(f"dpo: train_loss={RESULTS['dpo']['train_loss']} "
      f"time={wall_min:.1f} min peak_vram={peak_gb:.2f} GB "
      f"(on top of {best_sft})")
free_gpu(dpo_trainer, model, base)

best SFT adapter by measured eval_loss: lora (eval_loss=0.0053)
dpo training rows: 2000


Extracting prompt from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
20,0.367100
40,0.262200
60,0.203000
80,0.165900
100,0.123500
120,0.119000
140,0.108000
160,0.101100
180,0.086800
200,0.102400


adapter backed up to /content/drive/MyDrive/code-repair/adapters/dpo
dpo: train_loss=0.1496 time=15.2 min peak_vram=7.80 GB (on top of lora)
gpu allocated after cleanup: 3.11 GB


# Part E -- Benchmark on held-out HumanEval

Protocol:
- Evaluation set: the HumanEval-derived broken-code items (never used in
  training in any form). `N_EVAL` items are drawn with a fixed seed.
- **pass@1**: one greedy generation per item; the fix counts only if it
  passes the problem's real `check()` tests **executed in the Part A
  sandbox** (the same `executor.py`, imported, not reimplemented).
- **pass@3**: three samples at temperature 0.8; counts if any passes.
- RAG ablation: the same items are run with and without the retrieval
  context that the real Part C pipeline precomputed for each item, so the
  effect of RAG+GraphRAG is isolated from the model variable.

Deterministic, execution-based -- no LLM judging anywhere.

In [13]:
# ---- Part E, step 1: sandbox smoke check on this machine ----
good = run_tests("def add(a, b):\n    return a + b\n",
                 "def check(candidate):\n    assert candidate(1, 2) == 3\n",
                 entry_point="add")
bad = run_tests("def add(a, b):\n    return a - b\n",
                "def check(candidate):\n    assert candidate(1, 2) == 3\n",
                entry_point="add")
assert good.ok and not bad.ok
print("sandbox operational on this runtime (POSIX rlimit branch)")

sandbox operational on this runtime (POSIX rlimit branch)


In [14]:
# ---- Part E, step 2: generation + execution harness ----
import re
from concurrent.futures import ThreadPoolExecutor

N_EVAL = 150          # items per arm; raise to len(eval_items) for the full set
PASS_AT_K_SAMPLES = 3
MAX_NEW_TOKENS = 640
GEN_BATCH = 8

eval_subset = random.Random(23).sample(eval_items,
                                       min(N_EVAL, len(eval_items)))
print(f"evaluation subset: {len(eval_subset)} of {len(eval_items)} items")

_CODE_RE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)

def extract_code(text):
    blocks = _CODE_RE.findall(text)
    return (max(blocks, key=len) if blocks else text).strip()

def build_eval_prompts(items, use_context):
    prompts = []
    for item in items:
        messages = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",
                     "content": user_prompt(item, use_context=use_context)}]
        prompts.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True))
    return prompts

def generate(model, prompts, do_sample, temperature=0.8):
    outs = []
    tokenizer.padding_side = "left"
    for start in range(0, len(prompts), GEN_BATCH):
        batch = prompts[start:start + GEN_BATCH]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            gen = model.generate(
                **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=do_sample,
                temperature=temperature if do_sample else None,
                top_p=0.95 if do_sample else None,
                pad_token_id=tokenizer.pad_token_id)
        for i in range(len(batch)):
            new_tokens = gen[i][enc["input_ids"].shape[1]:]
            outs.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return outs

def execute_fix(args):
    item, fix_code = args
    r = run_tests(fix_code, item["test_code"],
                  entry_point=item["entry_point"], timeout_s=8.0)
    return r.ok

def evaluate_arm(tag, adapter=None, four_bit=False, use_context=False):
    print(f"===== eval arm: {tag} =====")
    t0 = time.monotonic()
    model = load_base(four_bit=four_bit)
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()
    model.config.use_cache = True

    prompts = build_eval_prompts(eval_subset, use_context)

    greedy = generate(model, prompts, do_sample=False)
    samples = [generate(model, prompts, do_sample=True)
               for _ in range(PASS_AT_K_SAMPLES)]
    free_gpu(model)

    with ThreadPoolExecutor(max_workers=8) as pool:
        greedy_ok = list(pool.map(execute_fix,
            [(it, extract_code(g)) for it, g in zip(eval_subset, greedy)]))
        sample_ok = []
        for s in samples:
            sample_ok.append(list(pool.map(execute_fix,
                [(it, extract_code(g)) for it, g in zip(eval_subset, s)])))

    n = len(eval_subset)
    pass1 = sum(greedy_ok) / n
    pass3 = sum(any(col) for col in zip(*sample_ok)) / n
    wall_min = (time.monotonic() - t0) / 60
    EVAL_RESULTS[tag] = {"pass@1": round(pass1, 4), "pass@3": round(pass3, 4),
                         "n": n, "minutes": round(wall_min, 1),
                         "rag_context": use_context}
    print(f"{tag}: pass@1={pass1:.3f} pass@3={pass3:.3f} "
          f"({n} items, {wall_min:.1f} min)")
    return EVAL_RESULTS[tag]

EVAL_RESULTS = {}

evaluation subset: 150 of 467 items


In [15]:
# ---- Part E, step 3: run every arm (each frees GPU before the next) ----
evaluate_arm("base")
evaluate_arm("lora", adapter=RESULTS["lora"]["adapter"])
evaluate_arm("qlora", adapter=RESULTS["qlora"]["adapter"], four_bit=True)
evaluate_arm("dora", adapter=RESULTS["dora"]["adapter"])
evaluate_arm("dpo", adapter=RESULTS["dpo"]["adapter"],
             four_bit=(RESULTS["dpo"]["base_sft"] == "qlora"))

===== eval arm: base =====


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


gpu allocated after cleanup: 5.98 GB
base: pass@1=0.687 pass@3=0.787 (150 items, 18.5 min)
===== eval arm: lora =====
gpu allocated after cleanup: 6.05 GB
lora: pass@1=0.933 pass@3=0.967 (150 items, 35.2 min)
===== eval arm: qlora =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


gpu allocated after cleanup: 4.72 GB
qlora: pass@1=0.927 pass@3=0.947 (150 items, 69.3 min)
===== eval arm: dora =====
gpu allocated after cleanup: 6.05 GB
dora: pass@1=0.933 pass@3=0.940 (150 items, 153.4 min)
===== eval arm: dpo =====
gpu allocated after cleanup: 6.05 GB
dpo: pass@1=0.500 pass@3=0.587 (150 items, 34.8 min)


{'pass@1': 0.5,
 'pass@3': 0.5867,
 'n': 150,
 'minutes': 34.8,
 'rag_context': False}

In [16]:
# ---- Part E, step 4: RAG ablation (same items, context on) ----
best_tag = max(("lora", "qlora", "dora", "dpo"),
               key=lambda k: EVAL_RESULTS[k]["pass@1"])
print(f"best arm by measured pass@1: {best_tag}")

evaluate_arm("base+RAG", use_context=True)
evaluate_arm(f"{best_tag}+RAG", adapter=RESULTS[best_tag]["adapter"],
             four_bit=(best_tag == "qlora" or
                       (best_tag == "dpo" and
                        RESULTS["dpo"]["base_sft"] == "qlora")),
             use_context=True)

best arm by measured pass@1: lora
===== eval arm: base+RAG =====
gpu allocated after cleanup: 5.98 GB
base+RAG: pass@1=0.693 pass@3=0.780 (150 items, 23.5 min)
===== eval arm: lora+RAG =====
gpu allocated after cleanup: 6.05 GB
lora+RAG: pass@1=0.887 pass@3=0.907 (150 items, 39.3 min)


{'pass@1': 0.8867,
 'pass@3': 0.9067,
 'n': 150,
 'minutes': 39.3,
 'rag_context': True}

In [17]:
# ---- Part E, step 5: combined results table (all values measured) ----
import pandas as pd

rows = []
for tag, m in EVAL_RESULTS.items():
    train = RESULTS.get(tag.replace("+RAG", ""), {})
    rows.append({
        "arm": tag,
        "pass@1": m["pass@1"],
        "pass@3": m["pass@3"],
        "RAG context": "yes" if m["rag_context"] else "no",
        "eval items": m["n"],
        "train eval_loss": train.get("eval_loss"),
        "train minutes": train.get("wall_minutes"),
        "train peak VRAM (GB)": train.get("peak_vram_gb"),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

table.to_csv("/content/benchmark_results.csv", index=False)
if "drive" in BACKUP_DIR:
    shutil.copy("/content/benchmark_results.csv",
                os.path.join(os.path.dirname(BACKUP_DIR),
                             "benchmark_results.csv"))
    print("results table copied to Drive")

     arm  pass@1  pass@3 RAG context  eval items  train eval_loss  train minutes  train peak VRAM (GB)
    base  0.6867  0.7867          no         150              NaN            NaN                   NaN
    lora  0.9333  0.9667          no         150           0.0053           23.5                 12.78
   qlora  0.9267  0.9467          no         150           0.0069           36.4                 11.03
    dora  0.9333  0.9400          no         150           0.0056           71.0                 12.79
     dpo  0.5000  0.5867          no         150              NaN           15.2                  7.80
base+RAG  0.6933  0.7800         yes         150              NaN            NaN                   NaN
lora+RAG  0.8867  0.9067         yes         150           0.0053           23.5                 12.78
results table copied to Drive


## Export: GGUF quantization

Merges the best-performing adapter into the base model, converts to GGUF,
and quantizes to q4_k_m for local serving with Ollama. The web UI's
`OLLAMA_MODEL` setting is a single config value pointing at whichever
model is registered.

In [18]:
# ---- merge the best adapter into the base model (bf16, on CPU RAM) ----
export_tag = best_tag
adapter_dir = RESULTS[export_tag.replace("+RAG", "")]["adapter"]
print(f"exporting arm: {export_tag} (adapter: {adapter_dir})")

merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu")
merged = PeftModel.from_pretrained(merge_base, adapter_dir)
merged = merged.merge_and_unload()
merged.save_pretrained("/content/merged", safe_serialization=True)
tokenizer.save_pretrained("/content/merged")
del merged, merge_base
gc.collect()
print("merged model saved to /content/merged")

exporting arm: lora (adapter: /content/adapters/lora)
merged model saved to /content/merged


In [19]:
# ---- build llama.cpp (installs cmake if missing) and quantize ----
import shutil as _sh
if _sh.which("cmake") is None:
    !apt-get -qq install -y cmake
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
%pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null

!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged \
    --outfile /content/code-repair-qwen-f16.gguf --outtype f16
!/content/llama.cpp/build/bin/llama-quantize \
    /content/code-repair-qwen-f16.gguf \
    /content/code-repair-qwen-q4_k_m.gguf q4_k_m

import os
for f in ("code-repair-qwen-f16.gguf", "code-repair-qwen-q4_k_m.gguf"):
    path = f"/content/{f}"
    if os.path.exists(path):
        print(f"{f}: {os.path.getsize(path) / 1024**3:.2f} GB")

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3570, done.
remote: Counting objects: 100% (3570/3570), done.
remote: Compressing objects: 100% (2841/2841), done.
remote: Total 3570 (delta 655), reused 3162 (delta 646), pack-reused 0 (from 0)
Receiving objects: 100% (3570/3570), 34.37 MiB | 17.29 MiB/s, done.
Resolving deltas: 100% (655/655), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account a

In [20]:
# ---- Modelfile + copy artifacts to Drive ----
MODELFILE = '''FROM ./code-repair-qwen-q4_k_m.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

PARAMETER stop <|im_end|>
PARAMETER temperature 0.2
SYSTEM You are a precise Python code repair assistant. Reply with the corrected, complete Python code in a single fenced code block and nothing else.
'''
with open("/content/Modelfile", "w") as f:
    f.write(MODELFILE)
print("wrote /content/Modelfile")

if "drive" in BACKUP_DIR:
    dest_dir = os.path.dirname(BACKUP_DIR)
    for f in ("code-repair-qwen-q4_k_m.gguf", "Modelfile"):
        _sh.copy(f"/content/{f}", os.path.join(dest_dir, f))
        print(f"copied {f} to {dest_dir}")
else:
    print("Drive not mounted -- download the GGUF and Modelfile manually "
          "from the Files panel before the runtime recycles.")

wrote /content/Modelfile
copied code-repair-qwen-q4_k_m.gguf to /content/drive/MyDrive/code-repair
copied Modelfile to /content/drive/MyDrive/code-repair


## Result

`code-repair-qwen-q4_k_m.gguf` and `Modelfile` are written to
`MyDrive/code-repair/`, built from the best-performing adapter by measured
pass@1 on the held-out benchmark. Registering it with Ollama
(`ollama create -f Modelfile`) and pointing `OLLAMA_MODEL` in
`ui/config.py` at it puts this exact fine-tuned model behind the web UI.